# Setup


In [ ]:
%matplotlib inline

import os, sys, warnings

import numpy as np
import scipy as sp
from scipy.spatial import distance
import pandas as pd
from tqdm import tqdm
import bct

import neurogym as ngym
import torch
import torch.nn as nn

from src.neural_network import RNN, run_testing
from src.utils import (normalize_x, build_reg_ken, fix_labels,
                       get_weight_masks_schaefer, get_file_str, get_my_colors)

import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='Arial')
import seaborn as sns
sns.set_style("white")
from src.plotting import my_reg_plot

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)


# Files and parameters


In [ ]:
# directories
username = os.getenv('USER')
if sys.platform == 'darwin':
    if username == 'ahmad':
        datadir = '/Users/ahmad/software/snaplab_github/neuro_rnn/data'
        modeldir = '/Volumes/Sabrent_2TB/rutgers/neuro_rnn/data/202506a'
        outdir = modeldir
elif sys.platform == 'linux':
    if username == 'ab2792':
        datadir = '/home/ab2792/software/snaplab_github/neuro_rnn/data'
        modeldir = '/home/ab2792/data/neuro_rnn/results/pytorch/model'
        outdir = modeldir
    elif username == 'lindenmp':
        datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
        modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/model'
        outdir = '/home/lindenmp/research_projects/neuro_rnn/results/figs'

save_figs = False

# data parameters
tasks = ['PerceptualDecisionMaking-v0', 'MultiSensoryIntegration-v0', 'ContextDecisionMaking-v0']
n_tasks = len(tasks)
dt = 100
batch_size = 32
decision = 400
seq_len_multi = 5

# RNN model and training parameters
rnn_model = 'rnn-tanh'
hidden_size = 100
n_runs = 25
n_epochs = 30000
lr = 0.001

# regularization parameters
reg_type = 'l2'
reg_weight = 0.002
mask_weights = True

# kernel types
kernel_types = ['sa_axis', 'euclidean', None]
kernel_labels = ['RNN-SA', 'RNN-E', 'RNN-Standard']
n_kernels = len(kernel_types)

# number of trials for lesion analysis
n_trials = 100


# Load data


In [ ]:
# weight masks
centroids = pd.read_csv(os.path.join(datadir, 'schaefer{0}_centroids.csv'.format(hidden_size * 2)))
centroids = centroids[:hidden_size]
roi_names = list(centroids['ROI Name'])
input_system = 'Vis'
output_system = 'Default'
masks = get_weight_masks_schaefer(roi_names=roi_names, input_system=input_system,
                                  output_system=output_system)
n_io = '{0}-{1}'.format(np.sum(masks['input_weight_mask']), np.sum(masks['output_weight_mask']))

# setup weight masks
input_weight_mask = np.zeros((hidden_size,))
output_weight_mask = np.zeros((hidden_size,))

# lesion masks
mask_labels = ['input_output_symmetric', 'input_bystanders_symmetric', 'output_bystanders_symmetric']
mask_labels_plot = ['input-output', 'input-internal', 'internal-output']
n_masks = len(mask_labels)
mask_list = [masks[ml] for ml in mask_labels]

# base config
config = {
    'datadir': datadir, 'outdir': outdir,
    'task': tasks[0], 'dt': dt, 'seq_len': 0, 'batch_size': batch_size,
    'rnn_model': rnn_model, 'hidden_size': hidden_size, 'n_runs': n_runs,
    'n_epochs': n_epochs, 'lr': lr, 'mask_weights': mask_weights,
    'reg_type': reg_type, 'reg_weight': reg_weight, 'kernel_type': kernel_types[0],
    'env_kwargs': {'dt': dt, 'timing': {'fixation': 200, 'stimulus': 1000,
                                         'delay': 0, 'decision': decision}},
    'n_io': n_io
}

# Device configuration
device = torch.device('cpu')

# get epoch list from first model
task = tasks[0]
if task == 'PerceptualDecisionMaking-v0':
    seq_len = 22
elif task == 'MultiSensoryIntegration-v0':
    seq_len = 11
elif task == 'ContextDecisionMaking-v0':
    seq_len = 13
seq_len = seq_len + int((decision - 100) / dt)
seq_len = seq_len * seq_len_multi
config['task'] = task
config['seq_len'] = seq_len
config['kernel_type'] = kernel_types[0]
file_str = get_file_str(config)

checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)
run = 0
epoch_list = []
N = 10
for idx in range(0, len(list(checkpoint[run].keys())), N):
    epoch_list.append(list(checkpoint[run].keys())[idx])
epoch_list = epoch_list[:21]
del checkpoint

n_logged_epochs = len(epoch_list)
print(f'Epochs: {n_logged_epochs}')
print(epoch_list)


# Edge group lesioning


Lesion input-output, input-internal, and internal-output edges and measure the accuracy drop.


In [ ]:
accuracy = np.zeros((n_tasks, n_kernels, n_runs, n_logged_epochs))
lesioned_accuracy = np.zeros((n_tasks, n_kernels, n_runs, n_logged_epochs, n_masks))

for i, task in enumerate(tasks):
    if task == 'PerceptualDecisionMaking-v0':
        seq_len = 22
    elif task == 'MultiSensoryIntegration-v0':
        seq_len = 11
    elif task == 'ContextDecisionMaking-v0':
        seq_len = 13
    seq_len = seq_len + int((decision - 100) / dt)
    seq_len = seq_len * seq_len_multi

    config['task'] = task
    config['seq_len'] = seq_len

    # setup dataset
    dataset = ngym.Dataset(config['task'], env_kwargs=config['env_kwargs'],
                           batch_size=config['batch_size'], seq_len=config['seq_len'])
    input_size = dataset.env.observation_space.shape[0]
    num_classes = dataset.env.action_space.n
    print(input_size, num_classes)

    for j, kernel_type in enumerate(kernel_types):
        config['kernel_type'] = kernel_type
        file_str = get_file_str(config)
        print(file_str)

        # load model checkpoint
        checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)

        # setup kernel
        regularization_kernel = None if kernel_type is None else np.zeros((hidden_size, hidden_size))

        # setup model
        model = RNN(input_size=input_size, hidden_size=hidden_size, num_classes=num_classes,
                    type=rnn_model, regularization_kernel=regularization_kernel,
                    input_weight_mask=input_weight_mask,
                    output_weight_mask=output_weight_mask).to(device)
        model.eval()

        # compute accuracy and lesioned accuracy
        for run in tqdm(np.arange(n_runs)):
            for e, epoch in enumerate(epoch_list):
                model.load_state_dict(checkpoint[run][epoch])
                accuracy[i, j, run, e], _, _, _, _, _ = run_testing(
                    dataset=dataset, model=model, n_trials=n_trials, verbose=False)

                for m, mask in enumerate(mask_list):
                    model.load_state_dict(checkpoint[run][epoch])
                    with torch.no_grad():
                        model.rnn.weight_hh_l0[mask] = 0
                    lesioned_accuracy[i, j, run, e, m], _, _, _, _, _ = run_testing(
                        dataset=dataset, model=model, n_trials=n_trials, verbose=False)
        del checkpoint


In [ ]:
color_palette = sns.color_palette("Set3")

# compute mean and CI
accuracy_mean = accuracy.mean(axis=2) * 100
lesioned_accuracy_mean = lesioned_accuracy.mean(axis=2) * 100
lesioned_accuracy_std = lesioned_accuracy.std(axis=2) * 100
ci = 1.96 * (lesioned_accuracy_std / np.sqrt(lesioned_accuracy.shape[2]))
ci_lower = lesioned_accuracy_mean - ci
ci_upper = lesioned_accuracy_mean + ci

for i, task in enumerate(tasks):
    fig_width = 8.5
    fig_height = 4
    f, ax = plt.subplots(1, n_kernels, figsize=(fig_width, fig_height))

    for j, kernel_type in enumerate(kernel_types):
        for m in np.arange(n_masks):
            jitter = m
            ax[j].plot(epoch_list,
                      lesioned_accuracy_mean[i, j, :, m] - accuracy_mean[i, j] - jitter,
                      color=color_palette[m], label=mask_labels_plot[m])
            ax[j].fill_between(epoch_list,
                              ci_lower[i, j, :, m] - accuracy_mean[i, j] - jitter,
                              ci_upper[i, j, :, m] - accuracy_mean[i, j] - jitter,
                              color=color_palette[m], alpha=0.25)

        sns.despine(offset=3, trim=False, left=False, right=True, top=True, bottom=False, ax=ax[j])
        ax[j].set_ylabel('Accuracy (%)')
        if j == 1:
            ax[j].set_title('Task: {0}\n{1}'.format(task, kernel_labels[j]))
        else:
            ax[j].set_title('\n{0}'.format(kernel_labels[j]))
        if j == (n_kernels - 1):
            ax[j].legend(bbox_to_anchor=(1, -0.2))

    f.tight_layout()
    plt.show()

    if save_figs:
        f.savefig(os.path.join(outdir, '{0}_lesion_node_groups.svg'.format(task)),
                  dpi=300, bbox_inches='tight', pad_inches=0.01)


# Random edge lesioning


In [ ]:
n_edges = hidden_size * hidden_size
lesion_percentages = np.linspace(0.01, 0.10, 10)
n_lesion_perc = len(lesion_percentages)
print(lesion_percentages)

lesioned_accuracy_rand = np.zeros((n_tasks, n_kernels, n_runs, n_lesion_perc))

for i, task in enumerate(tasks):
    if task == 'PerceptualDecisionMaking-v0':
        seq_len = 22
    elif task == 'MultiSensoryIntegration-v0':
        seq_len = 11
    elif task == 'ContextDecisionMaking-v0':
        seq_len = 13
    seq_len = seq_len + int((decision - 100) / dt)
    seq_len = seq_len * seq_len_multi

    config['task'] = task
    config['seq_len'] = seq_len

    dataset = ngym.Dataset(config['task'], env_kwargs=config['env_kwargs'],
                           batch_size=config['batch_size'], seq_len=config['seq_len'])
    input_size = dataset.env.observation_space.shape[0]
    num_classes = dataset.env.action_space.n

    for j, kernel_type in enumerate(kernel_types):
        config['kernel_type'] = kernel_type
        file_str = get_file_str(config)

        checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)

        regularization_kernel = None if kernel_type is None else np.zeros((hidden_size, hidden_size))

        model = RNN(input_size=input_size, hidden_size=hidden_size, num_classes=num_classes,
                    type=rnn_model, regularization_kernel=regularization_kernel,
                    input_weight_mask=input_weight_mask,
                    output_weight_mask=output_weight_mask).to(device)
        model.eval()

        epoch = epoch_list[-1]
        for run in tqdm(np.arange(n_runs)):
            for l, lesion_perc in enumerate(lesion_percentages):
                np.random.seed(42)
                mask = np.zeros((hidden_size, hidden_size)).astype(bool).reshape(n_edges)
                indices = np.random.choice(np.arange(mask.size), replace=False,
                                          size=int(n_edges * lesion_perc))
                mask[indices] = True
                mask = mask.reshape(hidden_size, hidden_size)

                model.load_state_dict(checkpoint[run][epoch])
                with torch.no_grad():
                    model.rnn.weight_hh_l0[mask] = 0
                lesioned_accuracy_rand[i, j, run, l], _, _, _, _, _ = run_testing(
                    dataset=dataset, model=model, n_trials=n_trials, verbose=False)
        del checkpoint


In [ ]:
fig_width = 8.5
fig_height = 6
f, ax = plt.subplots(n_tasks, n_kernels, figsize=(fig_width, fig_height))

for i, task in enumerate(tasks):
    for j, kernel_type in enumerate(kernel_types):
        df = pd.DataFrame(data=lesioned_accuracy_rand[i, j] * 100,
                          columns=np.round(lesion_percentages * 100).astype(int))
        sns.barplot(df, ax=ax[i, j], estimator='mean', errorbar='ci', palette='rocket')

        sns.despine(offset=3, trim=False, left=False, right=True, top=True, bottom=False,
                    ax=ax[i, j])
        ax[i, j].set_xlabel('Lesioned Edges (%)')
        ax[i, j].set_ylabel('Accuracy (%)')
        ax[i, j].set_ylim([0, 100])
        if j == 1:
            ax[i, j].set_title('{0}\n{1}'.format(task, kernel_type))
        else:
            ax[i, j].set_title('\n{0}'.format(kernel_type))

f.tight_layout()
plt.show()


# Single node lesioning


In [ ]:
# load a single model for single-node lesioning
config['task'] = tasks[0]
if tasks[0] == 'PerceptualDecisionMaking-v0':
    seq_len = 22
elif tasks[0] == 'MultiSensoryIntegration-v0':
    seq_len = 11
elif tasks[0] == 'ContextDecisionMaking-v0':
    seq_len = 13
seq_len = seq_len + int((decision - 100) / dt)
seq_len = seq_len * seq_len_multi
config['seq_len'] = seq_len
config['kernel_type'] = kernel_types[0]

file_str = get_file_str(config)
dataset = ngym.Dataset(config['task'], env_kwargs=config['env_kwargs'],
                       batch_size=config['batch_size'], seq_len=config['seq_len'])
input_size = dataset.env.observation_space.shape[0]
num_classes = dataset.env.action_space.n

regularization_kernel = np.zeros((hidden_size, hidden_size))
model = RNN(input_size=input_size, hidden_size=hidden_size, num_classes=num_classes,
            type=rnn_model, regularization_kernel=regularization_kernel,
            input_weight_mask=input_weight_mask,
            output_weight_mask=output_weight_mask).to(device)
model.eval()

checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)
epoch = epoch_list[-1]

# compute single-node lesioned accuracy
lesioned_accuracy_node = np.zeros((n_runs, hidden_size))

for run in tqdm(np.arange(n_runs)):
    for node in np.arange(hidden_size):
        model.load_state_dict(checkpoint[run][epoch])
        with torch.no_grad():
            model.rnn.weight_hh_l0[node, :] = 0
            model.rnn.weight_hh_l0[:, node] = 0
        lesioned_accuracy_node[run, node], _, _, _, _, _ = run_testing(
            dataset=dataset, model=model, n_trials=n_trials, verbose=False)


In [ ]:
f, ax = plt.subplots(1, 1, figsize=(5, 5))
sns.histplot(lesioned_accuracy_node[:, :].mean(axis=0), ax=ax)
ax.set_xlabel('Lesioned Accuracy')
ax.set_ylabel('Count')
ax.set_title('Single Node Lesion Impact')
plt.show()


In [ ]:
# correlation between node degree and lesion impact
network_stats = np.zeros((n_runs,))

for run in tqdm(np.arange(n_runs)):
    A = checkpoint[run][epoch]['rnn.weight_hh_l0'].detach().cpu().numpy().copy()
    A = np.abs(A)
    A = A / np.linalg.norm(A)
    thresh1 = np.quantile(A, q=0.9)
    mask = A > thresh1
    A[mask] = 1
    A[~mask] = 0
    degree = np.sum(A, axis=0)
    network_stats[run] = sp.stats.spearmanr(lesioned_accuracy_node[run, :], degree)[0]

f, ax = plt.subplots(1, 1, figsize=(5, 5))
sns.histplot(network_stats, ax=ax)
ax.set_xlabel('Spearman rho (degree vs. lesion accuracy)')
ax.set_ylabel('Count')
ax.set_title('Degree-Lesion Correlation Across Runs')
plt.show()
